In [1]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel

from langchain_google_vertexai import ChatVertexAI

import vertexai

# Vertex AI 서비스 연결 정보(프로젝트/리전) 설정
# 보통 노트북 시작 시 1회만 실행하면 됩니다.
vertexai.init(project="ai-prompt-evaluator-489612", location="us-central1")

# LangChain에서 Vertex AI Gemini 모델을 호출하는 LLM 객체
llm = ChatVertexAI(
  model_name="gemini-2.5-flash-lite",  # 빠르고 가벼운 Gemini 모델
  max_tokens=500  # 한 번에 생성할 최대 토큰 수
)

c:\my_study\workflow-architectures\.venv\Lib\site-packages\google\cloud\aiplatform\models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


In [2]:
class State(TypedDict):
  dish: str
  ingredients: list[dict]
  recipe_steps: str
  plating_instructions: str


class Ingredient(BaseModel):
  name: str
  quantity: str
  unit: str


class IngredientsOutput(BaseModel):
  ingredients: List[Ingredient]

In [ ]:
def list_ingredients(state: State):
  # 1) 모델 응답을 바로 Pydantic(IngredientsOutput) 형태로 받도록 설정
  structured_llm = llm.with_structured_output(IngredientsOutput)
  response = structured_llm.invoke(
    f"List 5-8 ingredients needed to make {state['dish']}"
  )

  # 2) 간혹 구조화 파싱이 실패하면(None), JSON 재요청으로 한 번 더 시도
  if response is None:
    fallback = llm.invoke(
      f"""
      Return ONLY valid JSON with this exact schema:
      {{
        "ingredients": [
          {{"name": "string", "quantity": "string", "unit": "string"}}
        ]
      }}

      List 5-8 ingredients needed to make {state['dish']}.
      No markdown, no explanation.
      """
    )

    # 3) 응답 본문(content)에서 JSON 객체 부분만 잘라서 파싱
    raw_text = fallback.content if hasattr(fallback, "content") else str(fallback)
    raw_text = raw_text.strip()
    json_start = raw_text.find("{")
    json_end = raw_text.rfind("}")

    if json_start == -1 or json_end == -1:
      raise ValueError("Could not parse JSON ingredients response from model")

    parsed = IngredientsOutput.model_validate_json(raw_text[json_start:json_end + 1])
    ingredients = parsed.ingredients
  else:
    ingredients = response.ingredients

  # 4) LangGraph state 타입에 맞게 Pydantic 객체 -> dict로 변환해서 반환
  return {
    "ingredients": [item.model_dump() for item in ingredients],
  }


def create_recipe(state: State):
  # 이전 노드(list_ingredients) 결과를 받아 레시피 단계 생성
  response = llm.invoke(
    f"Write a step by step cooking instruction for {state['dish']}, using these ingredients {state['ingredients']}",
  )

  return {
    "recipe_steps": response.content,
  }


def describe_plating(state: State):
  # 생성된 레시피를 바탕으로 플레이팅(담는 방법) 설명 생성
  response = llm.invoke(
    f"""
    Describe how to beautifully plate this dish {state['dish']} based on this recipe {state['recipe_steps']}
    한글로 설명 부탁해요.
    """,
  )

  return {
    "plating_instructions": response.content,
  }

In [4]:
graph_builder = StateGraph(State)

graph_builder.add_node("list_ingredients", list_ingredients)
graph_builder.add_node("create_recipe", create_recipe)
graph_builder.add_node("describe_plating", describe_plating)

graph_builder.add_edge(START, "list_ingredients")
graph_builder.add_edge("list_ingredients", "create_recipe")
graph_builder.add_edge("create_recipe", "describe_plating")
graph_builder.add_edge("describe_plating", END)

graph = graph_builder.compile()

In [6]:
# graph   # 그래프 출력

graph.invoke({
  "dish": "감바스"
})

{'dish': '감바스',
 'ingredients': [{'name': 'shrimp', 'quantity': '1', 'unit': 'pound'},
  {'name': 'garlic', 'quantity': '1', 'unit': 'head'},
  {'name': 'olive oil', 'quantity': '1', 'unit': 'cup'},
  {'name': 'red pepper flakes', 'quantity': '1', 'unit': 'teaspoon'},
  {'name': 'parsley', 'quantity': '1', 'unit': 'bunch'},
  {'name': 'baguette', 'quantity': '1', 'unit': 'loaf'}],
 'recipe_steps': "Here's a step-by-step recipe for delicious Gambas al Ajillo (Garlic Shrimp), using the ingredients you provided:\n\n**Yields:** 2-3 servings\n**Prep time:** 10 minutes\n**Cook time:** 15-20 minutes\n\n**Ingredients:**\n\n*   1 pound shrimp, peeled and deveined\n*   1 head garlic, thinly sliced\n*   1 cup olive oil\n*   1 teaspoon red pepper flakes (or more, to taste)\n*   1 bunch parsley, roughly chopped\n*   1 loaf baguette, sliced\n\n**Equipment You'll Need:**\n\n*   Large skillet or a shallow oven-safe cazuela (traditional Spanish clay pot)\n*   Knife and cutting board\n*   Small bowl (fo